<a href="https://colab.research.google.com/github/Faliirham/kids-dental-screening/blob/main/Skripsi_DentalCare_Qc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# SmileScan - QC dataset sebelum training (Google Colab)
# Output: clean_1024/, contact_sheets/, qc_report.csv, manual_review.csv
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

import os, re, math
from collections import defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, ImageFont
import cv2

DATA_DIR = '/content/drive/MyDrive/DATASET_GIGI'
OUT_DIR  = '/content/drive/MyDrive/DATASET_GIGI/DATASET_QC'
CLEAN_DIR = os.path.join(OUT_DIR, 'clean_1024')
SHEET_DIR = os.path.join(OUT_DIR, 'contact_sheets')
for d in (CLEAN_DIR, SHEET_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(DATA_DIR), f'Folder tidak ditemukan: {DATA_DIR}'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Ambang awal. Kalibrasi setelah lihat contact sheet; jangan dianggap final.
BLUR_REJECT, BLUR_WARN = 40, 80       # variance Laplacian @ sisi 1024 px
DARK, BRIGHT = 60, 200                # rata-rata grayscale 0-255
LOW_CONTRAST = 35
OVER_REJECT, OVER_WARN = 0.25, 0.10   # proporsi piksel >= 245
MIN_SIDE = 400                        # px sisi terpendek setelah orientasi
DUP_HAMMING = 4                       # 64-bit dHash; <=4 dianggap mirip
SHEET_PER_PAGE = 24


In [3]:
# ---------- 1. Manifest ----------
files = []
for root, _, names in os.walk(DATA_DIR):
    files += [os.path.join(root, n) for n in names
              if n.lower().endswith(('.jpg', '.jpeg', '.png'))]
files.sort()
print(f'File gambar ditemukan: {len(files)}')
assert files, 'Tidak ada file gambar di DATA_DIR.'

File gambar ditemukan: 186


In [4]:
# ---------- 2. Baca, koreksi orientasi, metrik ----------
pat = re.compile(r'(\d+)[-_](frontal|oklusal)', re.I)
try:
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    if face_cascade.empty():
        face_cascade = None
except Exception:
    face_cascade = None

rows = []
for idx, path in enumerate(files):
    fn = os.path.basename(path)
    stem = os.path.splitext(fn)[0]
    rel = os.path.relpath(path, DATA_DIR).replace(os.sep, '/')
    m = pat.search(stem)
    patient = m.group(1) if m else None
    view = m.group(2).lower() if m else None
    if view is None:
        parent = os.path.basename(os.path.dirname(path)).lower()
        view = next((v for v in ('frontal', 'oklusal') if v in parent), None)
    if patient is None:
        m2 = re.search(r'(\d+)', stem)
        patient = m2.group(1) if m2 else None

    rec = dict(filename=rel, source=path, patient=patient, view=view,
               width=None, height=None, exif_orientation=None,
               blur=None, brightness=None, contrast=None,
               overexposed=None, underexposed=None, faces=None,
               duplicate_of='', clean_path='', auto_verdict='PASS', reasons='')
    try:
        with Image.open(path) as im:
            im.verify()
        im = Image.open(path)
        rec['exif_orientation'] = im.getexif().get(274, 1)
        im = ImageOps.exif_transpose(im).convert('RGB')
        rec['width'], rec['height'] = im.size
        im.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
        arr = np.asarray(im)
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        rec['blur'] = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        rec['brightness'] = float(gray.mean())
        rec['contrast'] = float(gray.std())
        rec['overexposed'] = float((gray >= 245).mean())
        rec['underexposed'] = float((gray <= 20).mean())
        if face_cascade is not None:
            h, w = gray.shape
            faces = face_cascade.detectMultiScale(
                gray, 1.1, 5, minSize=(max(40, w // 12), max(40, h // 12)))
            rec['faces'] = len(faces)
        clean = os.path.join(CLEAN_DIR, f'{idx:03d}_{stem}.jpg')
        im.save(clean, quality=92)
        rec['clean_path'] = clean
    except Exception as e:
        rec['auto_verdict'] = 'REJECT'
        rec['reasons'] = f'corrupt: {type(e).__name__}'
    rows.append(rec)

print('Berhasil dibaca:', sum(1 for r in rows if r['clean_path']),
      '| gagal:', sum(1 for r in rows if not r['clean_path']))

Berhasil dibaca: 186 | gagal: 0


In [6]:
# ---------- 3. Verdict otomatis ----------
def judge(r):
    if r['auto_verdict'] == 'REJECT':
        return
    problems, hard = [], False
    def flag(msg, is_hard=False):
        nonlocal hard
        problems.append(msg)
        hard = hard or is_hard

    if r['width'] is None or min(r['width'], r['height']) < MIN_SIDE:
        flag('resolusi kecil', True)
    if r['blur'] < BLUR_REJECT:
        flag('sangat blur', True)
    elif r['blur'] < BLUR_WARN:
        flag('agak blur')
    if r['brightness'] < DARK:
        flag('terlalu gelap', True)
    elif r['brightness'] > BRIGHT:
        flag('terlalu terang', True)
    if r['contrast'] < LOW_CONTRAST:
        flag('kontras rendah')
    if r['overexposed'] > OVER_REJECT:
        flag('overexposed', True)
    elif r['overexposed'] > OVER_WARN:
        flag('sedikit overexposed')
    if not r['patient'] or not r['view']:
        flag('nama file tidak sesuai pola', True)

    r['reasons'] = '; '.join(problems)
    r['auto_verdict'] = 'REJECT' if hard else ('WARN' if problems else 'PASS')

for r in rows:
    judge(r)

In [7]:
# ---------- 4. Near-duplicate (dHash) ----------
def dhash(path):
    im = Image.open(path).convert('L').resize((9, 8), Image.Resampling.LANCZOS)
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int(''.join('1' if b else '0' for b in bits), 2)

by_name = {r['filename']: r for r in rows if r['clean_path']}
names = list(by_name)
hashes = {n: dhash(by_name[n]['clean_path']) for n in names}
near_dupes = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        dist = (hashes[a] ^ hashes[b]).bit_count()
        if dist <= DUP_HAMMING:
            near_dupes.append((a, b, dist))
            if not by_name[b]['duplicate_of']:
                by_name[b]['duplicate_of'] = a
                if by_name[b]['auto_verdict'] == 'PASS':
                    by_name[b]['auto_verdict'] = 'WARN'
                by_name[b]['reasons'] = '; '.join(
                    filter(None, [by_name[b]['reasons'], 'mirip duplikat']))
print('Pasangan near-duplicate:', len(near_dupes))

Pasangan near-duplicate: 0


In [8]:
# ---------- 5. Contact sheet ----------
def fmt(v, nd=0):
    return 'NA' if v is None else f'{v:.{nd}f}'

try:
    FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 14)
except Exception:
    FONT = ImageFont.load_default()

COLORS = {'PASS': (0, 128, 0), 'WARN': (190, 100, 0), 'REJECT': (190, 0, 0)}

def sheet(records, out_path, title):
    cols, tw, th, lh, head = 6, 300, 220, 48, 34
    nrows = math.ceil(len(records) / cols)
    canvas = Image.new('RGB', (cols * tw, head + nrows * (th + lh)), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), title, fill='black', font=FONT)
    for i, r in enumerate(records):
        x, y = (i % cols) * tw, head + (i // cols) * (th + lh)
        if r['clean_path'] and os.path.exists(r['clean_path']):
            with Image.open(r['clean_path']) as im:
                im.thumbnail((tw - 8, th - 8), Image.Resampling.LANCZOS)
                canvas.paste(im, (x + 4 + (tw - 8 - im.width) // 2,
                                  y + 4 + (th - 8 - im.height) // 2))
        color = COLORS.get(r['auto_verdict'], (0, 0, 0))
        label = r['filename'] if len(r['filename']) <= 38 else '...' + r['filename'][-35:]
        draw.text((x + 5, y + th), label, fill=color, font=FONT)
        draw.text((x + 5, y + th + 17),
                  f"{r['auto_verdict']}  blur={fmt(r['blur'])}  b={fmt(r['brightness'])}  "
                  f"c={fmt(r['contrast'])}  f={fmt(r['faces'])}",
                  fill='black', font=FONT)
        if r['reasons']:
            draw.text((x + 5, y + th + 33), r['reasons'][:44], fill=color, font=FONT)
    canvas.save(out_path, quality=88)

order = {'REJECT': 0, 'WARN': 1, 'PASS': 2}
for view in ('frontal', 'oklusal'):
    recs = [r for r in rows if r['view'] == view]
    recs.sort(key=lambda r: (order.get(r['auto_verdict'], 3),
                             r['blur'] if r['blur'] is not None else -1))
    for page in range(0, len(recs), SHEET_PER_PAGE):
        chunk = recs[page:page + SHEET_PER_PAGE]
        sheet(chunk,
              os.path.join(SHEET_DIR, f'{view}_{page // SHEET_PER_PAGE + 1:02d}.jpg'),
              f'{view.upper()} - halaman {page // SHEET_PER_PAGE + 1} (paling bermasalah dulu)')

rest = [r for r in rows if r['view'] not in ('frontal', 'oklusal')]
if rest:
    sheet(rest, os.path.join(SHEET_DIR, 'unparsed.jpg'), 'FILE TIDAK SESUAI POLA NAMA')

In [9]:
# ---------- 6. CSV + ringkasan ----------
df = pd.DataFrame(rows).drop(columns=['source'])
df['_order'] = df['auto_verdict'].map(order)
df = (df.sort_values(['view', '_order', 'blur'], na_position='first')
        .drop(columns=['_order']))
df.to_csv(os.path.join(OUT_DIR, 'qc_report.csv'), index=False)

manual = df[['filename', 'patient', 'view', 'auto_verdict', 'reasons',
             'blur', 'brightness', 'contrast', 'overexposed', 'faces',
             'duplicate_of']].copy()
manual['manual_ok'] = ''       # isi y/n: gigi terlihat, fokus, layak
manual['view_correct'] = ''    # isi y/n: label frontal/oklusal benar
manual['note'] = ''
manual.to_csv(os.path.join(OUT_DIR, 'manual_review.csv'), index=False)

print('\n=== RINGKASAN ===')
print('Total foto:', len(df))
print(df.groupby(['view', 'auto_verdict'], dropna=False).size().to_string())

pairs = defaultdict(lambda: defaultdict(list))
for r in rows:
    if r['patient'] and r['view']:
        pairs[r['patient']][r['view']].append(r['filename'])
missing = {p: dict(v) for p, v in pairs.items() if set(v) != {'frontal', 'oklusal'}}
print('\nPasangan tidak lengkap:', missing if missing else 'tidak ada')
multi = {p: {v: f for v, f in d.items() if len(f) > 1}
         for p, d in pairs.items() if any(len(f) > 1 for f in d.values())}
print('Patient/view ganda:', multi if multi else 'tidak ada')

print('\nNear-duplicate pairs:')
for a, b, d in near_dupes[:20]:
    print(f'  {a} <-> {b} (jarak {d})')
if len(near_dupes) > 20:
    print(f'  ... {len(near_dupes) - 20} lagi (lihat kolom duplicate_of)')

print('\nWajah terdeteksi:', int((df['faces'] > 0).sum()), '/', len(df),
      '(indikasi privasi; bukan penentu kelayakan)')
print(f'\nThreshold awal: blur < {BLUR_REJECT} reject / < {BLUR_WARN} warn; '
      f'brightness {DARK}-{BRIGHT}; overexposed > {OVER_REJECT}')
print('\nOutput:')
print(' ', CLEAN_DIR)
print(' ', SHEET_DIR)
print(' ', os.path.join(OUT_DIR, 'qc_report.csv'))
print(' ', os.path.join(OUT_DIR, 'manual_review.csv'))
print('\nLangkah berikut: buka contact_sheets, isi manual_review.csv, jalankan Cell 2.')


=== RINGKASAN ===
Total foto: 186
view     auto_verdict
frontal  PASS            73
         REJECT           2
         WARN            18
oklusal  PASS            49
         REJECT           8
         WARN            36

Pasangan tidak lengkap: tidak ada
Patient/view ganda: tidak ada

Near-duplicate pairs:

Wajah terdeteksi: 0 / 186 (indikasi privasi; bukan penentu kelayakan)

Threshold awal: blur < 40 reject / < 80 warn; brightness 60-200; overexposed > 0.25

Output:
  /content/drive/MyDrive/DATASET_GIGI/DATASET_QC/clean_1024
  /content/drive/MyDrive/DATASET_GIGI/DATASET_QC/contact_sheets
  /content/drive/MyDrive/DATASET_GIGI/DATASET_QC/qc_report.csv
  /content/drive/MyDrive/DATASET_GIGI/DATASET_QC/manual_review.csv

Langkah berikut: buka contact_sheets, isi manual_review.csv, jalankan Cell 2.


In [10]:
#Pengecekan Final
import os, pandas as pd

OUT_DIR = '/content/drive/MyDrive/DATASET_GIGI/DATASET_QC'
rev = pd.read_csv(os.path.join(OUT_DIR, 'manual_review.csv'),
                  keep_default_na=False, encoding='utf-8-sig')
rev['manual_ok'] = rev['manual_ok'].str.strip().str.lower()
rev['view_correct'] = rev['view_correct'].str.strip().str.lower()

ok = rev[(rev['manual_ok'] == 'y') & (rev['view_correct'] == 'y')].copy()
ok.to_csv(os.path.join(OUT_DIR, 'eligible.csv'), index=False)

print('Foto layak:', len(ok))
print(ok.groupby('view').size().to_string())
print('Belum diisi:', ((rev['manual_ok'] == '') | (rev['view_correct'] == '')).sum())

Foto layak: 0
Series([], )
Belum diisi: 186
